# 02. Derive trial metrics

Turns the event log into one row per trial using the proposal's definitions (3.2-3.3). **Dummy data by default.**

- A *selection* is a tap on a menu item; it is an **error** if the item is not on the correct path P*.
- An *error episode* starts at an error and ends when the current node is back on P* (an on-path tap, or a Back/collapse landing on-path); if the trial ends first, it ends at the trial end.
- **R** = sum of episode durations (Eq. 1); **T_find** = time of the terminal event; **T_clean = T_find - R** (Eq. 2).
- Give-up and time-out trials keep T_find = trial end but `success = 0`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

DATA_DIR = ROOT / "data" / "dummy"      # <- point this at data/raw/<your export folder> for real data
OUT_DIR = ROOT / "outputs"

from pipeline.prepare import prepare

prep = prepare(DATA_DIR)
trials, episodes = prep.trials, prep.episodes
trials.head()

,participant_id,trial_id,sequence,block_position,trial_order,pattern,task_id,task_pair,target_node,min_selections,t_end_s,end_reason,success,E,R,T_find,T_clean,n_selections,n_back_actions,selection_path,difficulty_rating,age,gender,gov_familiarity,used_reference_site,failure_type
0,P01,P01_T1,S1,1,1,DH,T1,A,C1.1.2,3,7.213,target,1,0,0.000,7.213,7.213,3,0,C1 > C1.1 > C1.1.2,5,20,F,2,N,
1,P01,P01_T2,S1,1,2,DH,T2,A,C2.2.1,3,16.759,target,1,2,6.132,16.759,10.627,5,2,C3 > C3.2 > C2 > C2.2 > C2.2.1,4,20,F,2,N,
2,P01,P01_T3,S1,2,3,MM,T3,B,C3.1.3,2,7.386,target,1,0,0.000,7.386,7.386,2,0,C3 > C3.1.3,7,20,F,2,N,
3,P01,P01_T4,S1,2,4,MM,T4,B,C4.3.2,2,7.283,target,1,0,0.000,7.283,7.283,2,0,C4 > C4.3.2,5,20,F,2,N,
4,P01,P01_T5,S1,3,5,ND,T5,C,C1.3.1,3,6.926,target,1,0,0.000,6.926,6.926,3,0,C1 > C1.3 > C1.3.1,5,20,F,2,N,


## Sanity checks
Eq. 2 must hold exactly, and the logged `on_path` flag should agree with the tree.

In [2]:
import numpy as np
print('T_find = T_clean + R for every trial:', bool(np.allclose(trials.T_find, trials.T_clean + trials.R)))
print('trials:', len(trials), '| with errors:', int((trials.E > 0).sum()), '| failed:', int((trials.success == 0).sum()))
print('validation: logged on_path mismatches =', 0 if 'on_path_matches_tree' not in prep.report.failed_names else 'see notebook 01')

T_find = T_clean + R for every trial: True
trials: 89 | with errors: 35 | failed: 4
validation: logged on_path mismatches = 0


## Look at one trial
Change `TRIAL_ID`. The table shows each tap and whether it was on the correct path; the episodes below it are what R is built from.

In [3]:
TRIAL_ID = trials[trials.E > 0].trial_id.iloc[0]
tree = prep.clean.tree
ev = prep.clean.events[prep.clean.events.trial_id == TRIAL_ID][['t_ms', 'event_type', 'node_id', 'current_node_after']].copy()
target = trials.loc[trials.trial_id == TRIAL_ID, 'target_node'].iloc[0]
ev['on_path_now'] = ev.current_node_after.isin(tree.path(target))
print('correct path:', ' > '.join(tree.path(target)))
ev

correct path: ROOT > C2 > C2.2 > C2.2.1


,t_ms,event_type,node_id,current_node_after,on_path_now
6,0,start,,ROOT,True
7,1096,menu_open,,ROOT,True
8,4499,select,C3,C3,False
9,7694,select,C3.2,C3.2,False
10,9787,back,,C3,False
11,10631,back,,ROOT,True
12,13159,select,C2,C2,True
13,14794,select,C2.2,C2.2,True
14,16759,select,C2.2.1,C2.2.1,True
15,16759,target_reached,C2.2.1,C2.2.1,True


In [4]:
print(trials.loc[trials.trial_id == TRIAL_ID, ['trial_id', 'pattern', 'E', 'R', 'T_find', 'T_clean', 'selection_path']].to_string(index=False))
episodes[episodes.trial_id == TRIAL_ID]

trial_id pattern  E     R  T_find  T_clean                 selection_path
  P01_T2      DH  2 6.132  16.759   10.627 C3 > C3.2 > C2 > C2.2 > C2.2.1


,trial_id,episode,start_s,end_s,duration_s,open_at_end
0,P01_T2,1,4.499,10.631,6.132,False


## Check against the generator's ground truth (dummy data only)
The dummy generator records the true E, R and T_find for each trial; the pipeline must reproduce them exactly.

In [5]:
gt_path = DATA_DIR / '_ground_truth_trials.csv'
if gt_path.exists():
    gt = pd.read_csv(gt_path)
    m = trials.merge(gt, on='trial_id', suffixes=('', '_true'))
    print('trials compared:', len(m))
    print('E equal:', bool((m.E == m.E_true).all()), '| R equal:', bool(np.allclose(m.R, m.R_true)), '| T_find equal:', bool(np.allclose(m.T_find, m.T_find_true)))
else:
    print('No ground-truth file (expected for real data).')

trials compared: 89
E equal: True | R equal: True | T_find equal: True
